# Defines a decision bondary for the 2D classification problem

In [ ]:
import os
import numpy as np
import pandas as pd 
from matplotlib import rcParams

import matplotlib.pyplot as plt

def setup_dirs(outDir):
    figuresDir = os.path.join(outDir, "figures")
    tablesDir = os.path.join(outDir, "tables")
    dataDir = os.path.join(outDir, "data")
    os.makedirs(dataDir, exist_ok=True)
    os.makedirs(figuresDir, exist_ok=True)
    os.makedirs(tablesDir, exist_ok=True)
    return figuresDir, tablesDir, dataDir

rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

outDir = '.'
figuresDir, tablesDir, dataDir = setup_dirs(outDir)


In [ ]:
DATA_DF_PATH = "path/to/your/data.csv"
df = pd.read_csv(DATA_DF_PATH)
df.columns

# filter by mean_cn > 6
df = df[df["mean_cn"] > 6].copy()
df_label = df[
    (df["mass_in_window"] > 0.4)
    & (df["mass_in_window"] < 0.8)
    & (df["score"] > 0.3)
    & (df["score"] < 0.6)
]

In [ ]:
def plot_scatter(df=None, output_path=None):
    # Plot a scatter plot: X: mass_in_window, Y: score, save to file
    plt.clf()
    plt.figure(figsize=(6,6))
    plt.scatter(df['mass_in_window'], df['score'], alpha=0.5)
    plt.xlabel('Mass in window')
    plt.ylabel('Score')
    plt.title('Scatter plot of Mass in window vs Score')
    plt.grid(True)
    plt.legend()
    plt.savefig(output_path, dpi=300)
    plt.close()

output_path = os.path.join(figuresDir, 'mass_vs_score_scatter_plot.png')
plot_scatter(df, output_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from adjustText import adjust_text


def get_process_color(df, proces_col="process"):
    processes = df[proces_col].unique()
    colors = plt.cm.get_cmap("tab10", len(processes))
    color_dict = {processes[i]: colors(i) for i in range(len(processes))}
    return df[proces_col].map(color_dict)


def get_training_anchors(df, q=0.1, small_value=0.02):
    x = df["mass_in_window"].values
    y = df["score"].values
    # Identify Extreme Points
    # We define labels purely based on your geometric quantiles
    x_left = np.quantile(x, q)
    y_high = np.quantile(y, 1 - q)
    # For the lower right
    x_right = np.quantile(x, 1 - q - small_value)
    y_low = np.quantile(y, q + small_value)
    # Class 0: Top-Left (Low Mass, High Score)
    mask_0 = (x < x_left) & (y > y_high)
    np.sum(mask_0)
    # Class 1: Bottom-Right (High Mass, Low Score)
    mask_1 = (x > x_right) & (y < y_low)
    np.sum(mask_1)
    # Combine these into a training set
    X_train = np.vstack(
        [
            np.column_stack([x[mask_0], y[mask_0]]),
            np.column_stack([x[mask_1], y[mask_1]]),
        ]
    )
    # Create labels (0 for Group A, 1 for Group B)
    y_train = np.concatenate([np.zeros(mask_0.sum()), np.ones(mask_1.sum())])
    # Give me the counts per class
    print(np.bincount(y_train.astype(int)))
    if len(np.unique(y_train)) < 2:
        raise ValueError(
            "Not enough extreme points found to form two classes. Increase q."
        )
    return X_train, y_train, mask_0, mask_1


def fit_lda(df=None, q=0.1):
    X_train, y_train, mask_0, mask_1 = get_training_anchors(df, q=q)
    # LDA finds the axis that maximizes separation relative to variance
    clf = LinearDiscriminantAnalysis(store_covariance=True)
    clf.fit(X_train, y_train)
    # Decision boundary is where w0*x + w1*y + b = 0
    w = clf.coef_[0]
    b = clf.intercept_[0]
    return clf, w, b, X_train, y_train, mask_0, mask_1


def fit_and_plot_lda_boundary(
    df,
    q=0.1,
    save_path=None,
    df_label=None,
    annotate_type=True,
    color_by_probability=False,
):
    """
    Fits LDA on extreme 'anchor' points and projects the boundary
    across the entire feature space.
    """
    clf, w, b, X_train, y_train, mask_0, mask_1 = fit_lda(df, q=q)
    x = df["mass_in_window"].values
    y = df["score"].values
    plt.figure(figsize=(6, 6))
    # 1. Plot all data (gray/neutral)
    sizes = np.log1p(df["n_cells"].values) * 10
    # If annotate_type is true, add a scatter_plot with just the boundary underneat the points with the process colors
    if annotate_type:
        plt.scatter(
            x,
            y,
            s=sizes + 0.1,
            c=get_process_color(df),
            alpha=0.5,
            edgecolors="none",
        )
    elif color_by_probability:
        # Get LDA scores for all data points, and convert to probabilities using logistic regression
        scores = clf.decision_function(np.column_stack([x, y]))
        scores = scores.reshape(-1, 1)
        cal = LogisticRegression(solver="lbfgs")
        cal.fit(X_train @ w.reshape(-1, 1), y_train)
        proba = cal.predict_proba(scores)[:, 1]
        # Plot with color based on probability
        plt.scatter(
            x,
            y,
            s=sizes,
            c=proba,
            cmap="viridis",
            alpha=0.8,
            label="Data (Colored by Probability)",
        )
    plt.scatter(x, y, s=sizes, color="lightgray", alpha=0.5, label="Unclassified Data")
    # 2. Highlight the Training Anchors (to show what LDA learned from)
    breakpoint()
    # Print how many in each class
    print(f"Training Class 0 (ecDNA): {np.sum(mask_0)} points")
    print(f"Training Class 1 (ICamp): {np.sum(mask_1) } points")
    plt.scatter(
        x[mask_0],
        y[mask_0],
        s=sizes[mask_0],
        color="orange",
        alpha=0.8,
        label="Training Class (ecDNA)",
    )
    plt.scatter(
        x[mask_1],
        y[mask_1],
        s=sizes[mask_1],
        color="purple",
        alpha=0.8,
        label="Training Class (ICamp)",
    )
    # 3. Draw the Decision Boundary
    # Equation: y = -(w0 * x + b) / w1
    x_grid = np.linspace(0, 1, 100)
    decision_boundary = -(w[0] * x_grid + b) / w[1]
    # Print the decision boundary line (i.e. y = mx + c)
    print(f"Decision boundary line: y = {-w[0]/w[1]:.3f}x + {-b/w[1]:.3f}")
    # 4. Draw Probability Bands (The "Principled" equivalent of your geometric band)
    # --- C2. Calibrate LDA score -> probability using 1D logistic regression ---
    # LDA score for training anchors
    score_train = X_train @ w + b
    score_train = score_train.reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs")
    cal.fit(score_train, y_train)

    def score_for_p(p):
        logit_p = np.log(p / (1.0 - p))
        return (logit_p - cal.intercept_[0]) / cal.coef_[0, 0]

    s_center = score_for_p(0.5)  # score where P=0.5
    s_hi = score_for_p(0.95)  # score where P=0.95
    s_lo = score_for_p(0.05)  # score where P=0.05
    # --- D. Plotting the calibrated boundaries ---
    x_grid = np.linspace(0, 1, 100)
    decision_boundary = -(w[0] * x_grid + b - s_center) / w[1]
    boundary_upper = -(w[0] * x_grid + b - s_hi) / w[1]
    boundary_lower = -(w[0] * x_grid + b - s_lo) / w[1]
    print(
        f"Decision boundary (P=0.5): y = {-w[0]/w[1]:.3f}x + {-(b - s_center)/w[1]:.3f}"
    )
    print(f"Upper boundary (P=0.95): y = {-w[0]/w[1]:.3f}x + {-(b - s_hi)/w[1]:.3f}")
    print(f"Lower boundary (P=0.05): y = {-w[0]/w[1]:.3f}x + {-(b - s_lo)/w[1]:.3f}")
    plt.plot(
        x_grid, decision_boundary, "k--", linewidth=2, label="LDA boundary (P=0.5)"
    )
    # Give them different colors
    plt.plot(
        x_grid,
        boundary_upper,
        "k:",
        linewidth=1,
        alpha=0.6,
        label="P=0.95",
        color="blue",
    )
    plt.plot(
        x_grid,
        boundary_lower,
        "k:",
        linewidth=1,
        alpha=0.6,
        label="P=0.05",
        color="red",
    )
    ########################################################
    # Now plot a line using the following slope and intercept
    slope = 0.859
    intercept = -0.184
    plt.xlim(-0.1, 1.1)
    plt.ylim(-0.1, 1.1)
    plt.xlabel("Mass in Window")
    plt.ylabel("ecDNA Score")
    # If df_label is provided, add annotations
    if df_label is not None:
        texts = []
        for _, row in df_label.iterrows():
            t = plt.text(
                row["mass_in_window"],
                row["score"],
                str(row["individual"]),
                fontsize=4,
            )
            texts.append(t)
        adjust_text(texts, arrowprops=dict(arrowstyle="->", color="r"))
    plt.legend(loc="upper right", fontsize="small")
    plt.title(f"LDA Boundary (Trained on q={q} extremes)")
    plt.grid(True, linestyle=":", alpha=0.6)
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

In [ ]:
q = 0.05
annotate_type = False
fit_and_plot_lda_boundary(
    df,
    q=q,
    save_path=os.path.join(
        figuresDir,
        f"lda_decision_boundary_q{q}_annot_type{annotate_type}_calibrated.pdf",
    ),
    df_label=None,
    annotate_type=annotate_type,
)
